# Figure 6: Redox Variables Along Degassing Paths (P-normalized)

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    PLOTLY_TICK_LEN,
    TOOL_COLORS_HEX
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]

# Redox-capable tools only (must have S6St_m in their standardized output).
TOOLS = ["EVo", "VolFe", "MAGEC", "DCompress", "DCompress (IM)", "SulfurX"]

Y_ROWS = [
    ("dFMQ",     "log<sub>10</sub>f O<sub>2</sub> (ΔFMQ)"),
    ("Fe3Fet_m", "Fe<sup>3+</sup>/Fe<sup>T</sup>"),
    ("S6St_m",   "S<sup>6+</sup>/S<sup>T</sup>"),
]


In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)
# Show which (sample, tool) pairs carry speciated-S data (required for row 3)

## Build the figure


In [ ]:
n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, vertical_spacing=0.03, horizontal_spacing=0.05,
    subplot_titles=subplot_titles,
)

for r, (col_name, y_label) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        for tool in TOOLS:
            df = systems.get(sample, {}).get(tool)
            if df is None or col_name not in df.columns or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            x_norm = df["P_bars"] / p_init
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=x_norm, y=df[col_name],
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2),
                    showlegend=(r == 1 and c == 1),
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label if c == 1 else None, row=r, col=c)
        if r == n_rows:
            fig.update_xaxes(title_text="P / P<sub>i</sub>", row=r, col=c, range=[0, None])

# logfO2 y-axis lims
for c in range(1, n_cols + 1):
    fig.update_yaxes(range=[-1.4, 3], row=1, col=c)

# Fe3/FeT y-axis lims
for c in range(1, n_cols + 1):
    fig.update_yaxes(range=[0.07, 0.5], row=2, col=c)

# S6+/ST y-axis 0-1
for c in range(1, n_cols + 1):
    fig.update_yaxes(range=[0, 1], row=3, col=c)

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=0.99, y=0.02,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=800, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)

if SAVE_FIG:
    fig.write_image("figures/Fig_redox_variables.png", scale=4)

fig.show()
